In [1]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

### Question 1: Frequency Distribution of Correct Answers

In [3]:
answer_counts = train_df['answer'].value_counts()
print("Frequency distribution of correct answers:")
print(answer_counts)

most_frequent_count = answer_counts.max()
least_frequent_count = answer_counts.min()
sum_of_counts = most_frequent_count + least_frequent_count

print(f"\nMost frequent option count: {most_frequent_count}")
print(f"Least frequent option count: {least_frequent_count}")
print(f"Answer: {sum_of_counts}")

Frequency distribution of correct answers:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent option count: 490
Least frequent option count: 324
Answer: 814


### Question 2: Unique Words in Cleaned Prompt Column

In [4]:
# Convert to lowercase and remove punctuation
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# Split by whitespace and get unique words
all_words = ' '.join(train_df['cleaned_prompt']).split()
unique_words = set(all_words)
vocabulary_size = len(unique_words)

print(f"Unique words in the cleaned prompt: {vocabulary_size}")

Unique words in the cleaned prompt: 859


### Question 3: Filter Stop Words from Prompt of Row ID 1

In [5]:
# Find the prompt for Row ID 1 (which has 'id' == 1 in the 'id' column)
prompt_row_id_1 = train_df[train_df['id'] == 1]['cleaned_prompt'].iloc[0]

# Split the prompt into words
words_row_id_1 = prompt_row_id_1.split()

# Filter out English stop words
filtered_words_row_id_1 = [word for word in words_row_id_1 if word not in ENGLISH_STOP_WORDS]

# Count the remaining words
num_words_after_filtering = len(filtered_words_row_id_1)

print(f"Cleaned prompt for Row ID 1: '{prompt_row_id_1}'")
print(f"Number of words left in the prompt for Row ID 1 after filtering stop words: {num_words_after_filtering}")

Cleaned prompt for Row ID 1: 'pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options'
Number of words left in the prompt for Row ID 1 after filtering stop words: 13


### Question 4: TF-IDF Vocabulary Size

In [6]:
# Combine all prompt and option texts into a single list
combined_texts = []
for index, row in train_df.iterrows():
    combined_texts.append(row['prompt'])
    combined_texts.append(row['A'])
    combined_texts.append(row['B'])
    combined_texts.append(row['C'])
    combined_texts.append(row['D'])
    combined_texts.append(row['E'])

# Initialize TfidfVectorizer with stop_words='english'
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# Fit the vectorizer on the combined text
tfidf_vectorizer.fit(combined_texts)

# Get the total number of feature columns (vocabulary size)
vocabulary_size_tfidf = len(tfidf_vectorizer.vocabulary_)

print(f"Vocabulary size generated by the TF-IDF vectorizer: {vocabulary_size_tfidf}")

Vocabulary size generated by the TF-IDF vectorizer: 2762


### Question 5: Cosine Similarity between Prompt and Option A for Row ID 1

In [7]:
# Get the prompt and option A for Row ID 1
row_id_1_data = train_df[train_df['id'] == 1]
prompt_row_id_1_q5 = row_id_1_data['prompt'].iloc[0]
option_a_row_id_1 = row_id_1_data['A'].iloc[0]

# Transform the prompt and option A using the fitted TF-IDF vectorizer
prompt_vector = tfidf_vectorizer.transform([prompt_row_id_1_q5])
option_a_vector = tfidf_vectorizer.transform([option_a_row_id_1])

# Calculate cosine similarity
cosine_sim = cosine_similarity(prompt_vector, option_a_vector)[0][0]

# Round to 4 decimal places
rounded_cosine_sim = round(cosine_sim, 4)

print(f"Cosine similarity between prompt and Option A for Row ID 1: {rounded_cosine_sim}")

Cosine similarity between prompt and Option A for Row ID 1: 0.2328


### Question 6: Percentage of Highest Cosine Similarity Matching Correct Answer

In [8]:
correct_predictions = 0
total_rows = len(train_df)

option_cols = ['A', 'B', 'C', 'D', 'E']

for index, row in train_df.iterrows():
    prompt_text = row['prompt']
    correct_answer_label = row['answer']

    # Vectorize the prompt
    prompt_vector = tfidf_vectorizer.transform([prompt_text])

    similarities = {}
    for option_label in option_cols:
        option_text = row[option_label]
        option_vector = tfidf_vectorizer.transform([option_text])
        sim = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities[option_label] = sim

    # Find the option with the highest similarity
    highest_sim_option = max(similarities, key=similarities.get)

    if highest_sim_option == correct_answer_label:
        correct_predictions += 1

percentage_match = (correct_predictions / total_rows) * 100

print(f"Number of instances where highest cosine similarity option matches correct answer: {correct_predictions}")
print(f"Total rows: {total_rows}")
print(f"Percentage of highest cosine similarity matching the correct answer: {percentage_match:.2f}%")

Number of instances where highest cosine similarity option matches correct answer: 274
Total rows: 2000
Percentage of highest cosine similarity matching the correct answer: 13.70%


### Question 7: MAP@3 Score for specific prediction (Ground truth C, Prediction C A B)

In [9]:
def calculate_map3(ground_truth, predictions):
    if not predictions:
        return 0.0

    score = 0.0
    num_hits = 0
    for i, pred in enumerate(predictions):
        if pred == ground_truth:
            num_hits += 1
            score += num_hits / (i + 1)

    # If the ground truth is not in the predictions, the score is 0
    if ground_truth not in predictions:
        return 0.0
    else:
        return score # Since there's only one relevant item


ground_truth_q7 = 'C'
predictions_q7 = ['C', 'A', 'B']

map3_score_q7 = calculate_map3(ground_truth_q7, predictions_q7)

print(f"Ground truth: {ground_truth_q7}")
print(f"Predictions: {predictions_q7}")
print(f"MAP@3 score: {map3_score_q7:.4f}")

Ground truth: C
Predictions: ['C', 'A', 'B']
MAP@3 score: 1.0000


### Question 8: MAP@3 Score for specific prediction (Ground truth B, Prediction D B E)

In [10]:
ground_truth_q8 = 'B'
predictions_q8 = ['D', 'B', 'E']

map3_score_q8 = calculate_map3(ground_truth_q8, predictions_q8)

print(f"Ground truth: {ground_truth_q8}")
print(f"Predictions: {predictions_q8}")
print(f"MAP@3 score: {map3_score_q8:.4f}")

Ground truth: B
Predictions: ['D', 'B', 'E']
MAP@3 score: 0.5000


### Question 9: Majority Class Baseline MAP@3 Score

In [11]:
# Find the top 3 most frequent answers
most_frequent_answers = answer_counts.nlargest(3).index.tolist()

print(f"Most frequent answers (in order): {most_frequent_answers}")

# Define the static predictions for the Majority Class Baseline
static_predictions = most_frequent_answers

# Calculate MAP@3 for each row and then average
total_map3_score = 0.0
for index, row in train_df.iterrows():
    ground_truth = row['answer']
    total_map3_score += calculate_map3(ground_truth, static_predictions)

overall_map3_majority_baseline = total_map3_score / len(train_df)

print(f"Overall MAP@3 score of the Majority Class Baseline: {overall_map3_majority_baseline:.4f}")

Most frequent answers (in order): ['B', 'C', 'A']
Overall MAP@3 score of the Majority Class Baseline: 0.4213


### Question 10: TF-IDF Pipeline Average MAP@3 Score

In [12]:
total_map3_score_tfidf_pipeline = 0.0

option_cols = ['A', 'B', 'C', 'D', 'E']

for index, row in train_df.iterrows():
    prompt_text = row['prompt']
    ground_truth = row['answer']

    # Vectorize the prompt
    prompt_vector = tfidf_vectorizer.transform([prompt_text])

    similarities = {}
    for option_label in option_cols:
        option_text = row[option_label]
        option_vector = tfidf_vectorizer.transform([option_text])
        sim = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities[option_label] = sim

    # Sort options by similarity in descending order to get top 3 predictions
    sorted_predictions = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    top_3_predictions = [option for option, sim in sorted_predictions[:3]]

    # Calculate MAP@3 for the current row
    total_map3_score_tfidf_pipeline += calculate_map3(ground_truth, top_3_predictions)

# Calculate the average MAP@3 score
average_map3_tfidf_pipeline = total_map3_score_tfidf_pipeline / len(train_df)

print(f"Total MAP@3 score for TF-IDF Pipeline: {total_map3_score_tfidf_pipeline}")
print(f"Average MAP@3 score of the TF-IDF Pipeline: {average_map3_tfidf_pipeline:.4f}")

Total MAP@3 score for TF-IDF Pipeline: 623.8333333333331
Average MAP@3 score of the TF-IDF Pipeline: 0.3119
